In [ ]:
import sys
!{sys.executable} -m pip install -U spacy
!{sys.executable} -m spacy download fr_core_news_sm

In [ ]:
!pip install tensorflow


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
Traceback (most recent call last):
  File "C:\Users\jhaifa\AppData\Roaming\Python\Python314\site-packages\urllib3\connectionpool.py", line 470, in _make_request
    self._validate_conn(conn)
    ~~~~~~~~~~~~~~~~~~~^^^^^^
  File "C:\Users\jhaifa\AppData\Roaming\Python\Python314\site-packages\urllib3\connectionpool.py", line 1125, in _validate_conn
    conn.connect()
    ~~~~~~~~~~~~^^
  File "C:\Users\jhaifa\AppData\Roaming\Python\Python314\site-packages\urllib3\connection.py", line 878, in connect
    wrapped_socket, is_verified = _ssl_wrap_socket_and_match_hostname(
                                  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        sock=sock,
        ^^^^^^^^^^
    ...<14 lines>...
        assert_fingerprint=self.assert_fingerprint,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\jhaifa\AppData\Roaming\Python\Python31

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
!pip install spacy
!python -m spacy download fr_core_news_sm

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
Traceback (most recent call last):
  File "C:\Users\jhaifa\AppData\Roaming\Python\Python314\site-packages\urllib3\connectionpool.py", line 470, in _make_request
    self._validate_conn(conn)
    ~~~~~~~~~~~~~~~~~~~^^^^^^
  File "C:\Users\jhaifa\AppData\Roaming\Python\Python314\site-packages\urllib3\connectionpool.py", line 1125, in _validate_conn
    conn.connect()
    ~~~~~~~~~~~~^^
  File "C:\Users\jhaifa\AppData\Roaming\Python\Python314\site-packages\urllib3\connection.py", line 878, in connect
    wrapped_socket, is_verified = _ssl_wrap_socket_and_match_hostname(
                                  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        sock=sock,
        ^^^^^^^^^^
    ...<14 lines>...
        assert_fingerprint=self.assert_fingerprint,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\jhaifa\AppData\Roaming\Python\Python31

## 1. Configuration

In [ ]:
import os
import json
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (f1_score, accuracy_score, precision_score, recall_score,
                              classification_report, confusion_matrix, ConfusionMatrixDisplay)
import joblib
import spacy

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models, callbacks
tf.get_logger().setLevel("ERROR")
try:
    import torch
    from torch import nn
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
    from datasets import Dataset
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_CSV = "corpus.csv"
ARTIFACTS_DIR = "artifacts"
WEBAPP_DIR = "webapp_export"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(WEBAPP_DIR, exist_ok=True)

### Prétraitement linguistique : spaCy

Les mots-outils français utilisés par la vectorisation TF-IDF viennent de **spaCy** (`fr_core_news_sm`) plutôt que d'une liste codée à la main : une liste vérifiée et maintenue par la communauté plutôt qu'une liste artisanale forcément incomplète. Un comparatif rapide (texte brut vs texte lemmatisé avec spaCy) a été mené en amont de ce notebook : la lemmatisation n'apportait pas de gain sur ce corpus (0.90 contre 0.92 de macro-F1 en validation croisée rapide) et complique l'export navigateur puisqu'aucun lemmatiseur léger n'existe côté JavaScript pour le français. Le texte brut (minuscule + stopwords spaCy) est donc conservé.

In [14]:
_NLP = spacy.load("fr_core_news_sm", disable=["parser", "ner"])
SPACY_STOPWORDS = sorted(_NLP.Defaults.stop_words)
print(f"{len(SPACY_STOPWORDS)} mots-outils français chargés depuis spaCy (fr_core_news_sm).")

OSError: [E050] Can't find model 'fr_core_news_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

## 2. Chargement du dataset

Le corpus (`corpus.csv`, colonnes `text,label`) a été **rédigé à la main**, message par message : formulations courtes et longues, registre formel et informel, fautes et abréviations réalistes.

In [2]:
def load_corpus(path=DATA_CSV):
    df = pd.read_csv(path)
    assert "text" in df.columns and "label" in df.columns, (
        f"Le CSV doit contenir les colonnes 'text' et 'label' (trouvé : {list(df.columns)})"
    )
    df = df.dropna(subset=["text", "label"]).reset_index(drop=True)
    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = df["label"].astype(str).str.strip()
    return df[df["text"].str.len() > 0].reset_index(drop=True)


def get_classes(df):
    # Ordre canonique des classes pour tout le pipeline : déduit des données
    # elles-memes (ordre alphabetique des labels presents), jamais d'une
    # liste codee en dur -- un autre CSV donnerait automatiquement un autre
    # ensemble de classes, sans toucher au code.
    return sorted(df["label"].unique().tolist())


df = load_corpus()
CLASSES = get_classes(df)
print(f"{len(df)} messages chargés, {len(CLASSES)} classes détectées automatiquement :")
print(CLASSES)
df.sample(8, random_state=3)

NameError: name 'DATA_CSV' is not defined

## 3. Analyse du dataset

Le sujet impose un jeu de classes déséquilibré et prévient que le choix des métriques doit en tenir compte. On vérifie ici l'ampleur réelle du déséquilibre, ainsi que la variété des longueurs de messages (le corpus contient volontairement des messages très courts, façon texto, et des messages longs, façon email détaillé).

In [ ]:
counts = df["label"].value_counts().reindex(CLASSES)

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(counts.index, counts.values, color="#4C72B0")
ax.set_ylabel("Nombre de messages")
ax.set_title("Répartition des messages par service (corpus rédigé à la main)")
plt.xticks(rotation=35, ha="right")
for b in bars:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 2, str(int(b.get_height())), ha="center", fontsize=9)
plt.tight_layout()
plt.show()

print(f"Classe majoritaire / minoritaire : {counts.max()} / {counts.min()} = ratio {counts.max()/counts.min():.1f}x")

In [ ]:
n_mots = df["text"].str.split().apply(len)
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(n_mots, bins=20, color="#55A868", edgecolor="white")
ax.set_xlabel("Nombre de mots par message")
ax.set_ylabel("Nombre de messages")
ax.set_title("Distribution de la longueur des messages (courts et longs)")
plt.tight_layout()
plt.show()
print(n_mots.describe().round(1))
print("\nExemple de message court :", df.loc[n_mots.idxmin(), "text"])
print("Exemple de message long   :", df.loc[n_mots.idxmax(), "text"][:200], "...")

## 4. Préparation des données

Chaque famille de modèle a son propre prétraitement  :

| Famille | Prétraitement |
|---|---|
| TF-IDF | minuscule, stopwords spaCy, n-grammes (1,2), pondération sublinéaire |
| BiLSTM | tokenizer Keras (indices de mots), padding/troncature |
| CamemBERT / DistilCamemBERT | tokenizer sous-mots du modèle pré-entraîné |

Pour que la comparaison reste équitable, **aucune donnée de validation ou de test n'est jamais ré-échantillonnée pour équilibrer les classes** : le déséquilibre observé en section 3 est conservé tel quel partout, et c'est aux modèles (via `class_weight`, pondération de la loss, etc.) de s'y adapter — pas aux données d'être truquées.

In [ ]:
def align_proba(proba, fitted_classes, target_classes):
    # Réordonne les colonnes de proba pour correspondre exactement à l'ordre
    # de `target_classes`, quel que soit l'ordre interne du modèle.
    idx = [list(fitted_classes).index(c) for c in target_classes]
    return proba[:, idx]

## 5. Split développement / test

80% développement (utilisé pour la validation croisée et la sélection du modèle) / 20% test final, . Split stratifié pour que même les classes les plus rares restent représentées dans les deux ensembles.

In [ ]:
def split_dev_test(df, test_size=0.2, seed=SEED):
    dev_df, test_df = train_test_split(df, test_size=test_size, random_state=seed, stratify=df["label"])
    return dev_df.reset_index(drop=True), test_df.reset_index(drop=True)


dev_df, test_df = split_dev_test(df)
print(f"Développement : {len(dev_df)} messages   |   Test final (de côté) : {len(test_df)} messages")
print("\nRépartition (développement) :")
print(dev_df["label"].value_counts().reindex(CLASSES))
print("\nRépartition (test final) :")
print(test_df["label"].value_counts().reindex(CLASSES))

## 6. TF-IDF + Logistic Regression

La famille TF-IDF est déclinée en **5 candidats** (LogisticRegression, LinearSVC calibré, MultinomialNB, ComplementNB, RandomForest), tous construits avec exactement la même vectorisation. Chacun expose l'interface générique de la section 4. `LogisticRegression` sert de démonstration illustrative ci-dessous.

In [ ]:
def make_tfidf_pipeline(clf):
    return Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=False, stop_words=SPACY_STOPWORDS,
                                   ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("clf", clf),
    ])


def tfidf_build_and_fit(clf_factory):
    def build_and_fit(train_texts, train_labels, classes):
        pipe = make_tfidf_pipeline(clf_factory())
        pipe.fit(train_texts.str.lower(), train_labels)
        return pipe
    return build_and_fit


def tfidf_predict_proba(fitted, texts, classes):
    proba = fitted.predict_proba(texts.str.lower())
    return align_proba(proba, fitted.classes_, classes)


def tfidf_save(fitted, out_dir, name):
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"{name}.joblib")
    joblib.dump(fitted, path)
    return {"model": path}


def tfidf_size(fitted):
    import io
    buf = io.BytesIO()
    joblib.dump(fitted, buf)
    return buf.tell()


def make_tfidf_candidate(name, clf_factory):
    safe = name.replace(" ", "_").replace("(", "").replace(")", "")
    return {
        "name": name, "family": "tfidf", "runnable": True, "reason_not_runnable": None,
        "build_and_fit": tfidf_build_and_fit(clf_factory),
        "predict_proba": lambda fitted, texts, classes: tfidf_predict_proba(fitted, texts, classes),
        "save": lambda fitted, out_dir: tfidf_save(fitted, out_dir, safe),
        "model_size_bytes": tfidf_size,
    }


TFIDF_CANDIDATES = [
    make_tfidf_candidate("TF-IDF + LogisticRegression",
                          lambda: LogisticRegression(max_iter=3000, class_weight="balanced", C=5.0, random_state=SEED)),
    make_tfidf_candidate("TF-IDF + LinearSVC (calibré)",
                          lambda: CalibratedClassifierCV(LinearSVC(class_weight="balanced", C=1.0, random_state=SEED), cv=3)),
    make_tfidf_candidate("TF-IDF + MultinomialNB", lambda: MultinomialNB(alpha=0.3)),
    make_tfidf_candidate("TF-IDF + ComplementNB", lambda: ComplementNB(alpha=0.3)),
    make_tfidf_candidate("TF-IDF + RandomForest",
                          lambda: RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=-1)),
]

# Démonstration illustrative (LogisticRegression) sur le split dev/test simple
demo = TFIDF_CANDIDATES[0]
t0 = time.time()
fitted_demo = demo["build_and_fit"](dev_df["text"], dev_df["label"], CLASSES)
print(f"Entraînement TF-IDF + LogisticRegression : {time.time()-t0:.2f}s")
proba_demo = demo["predict_proba"](fitted_demo, test_df["text"], CLASSES)
preds_demo = [CLASSES[i] for i in proba_demo.argmax(axis=1)]
print(f"Accuracy (démonstration, test final) : {accuracy_score(test_df['label'], preds_demo):.3f}")
print(f"Macro-F1 (démonstration, test final) : {f1_score(test_df['label'], preds_demo, average='macro'):.3f}")
print(f"Taille du modèle : {demo['model_size_bytes'](fitted_demo)/1024:.0f} Ko")

## 7. BiLSTM

Embeddings entraînés *from scratch* (pas de poids pré-entraînés téléchargés ), `Bidirectional(LSTM)`, `class_weight` pour le déséquilibre, arrêt anticipé sur la perte de validation.

In [ ]:
MAX_WORDS, EMBED_DIM, LSTM_UNITS = 4000, 64, 32


def build_bilstm_model(vocab_size, maxlen, n_classes):
    m = models.Sequential([
        layers.Input(shape=(maxlen,)),
        layers.Embedding(input_dim=vocab_size, output_dim=EMBED_DIM, mask_zero=True),
        layers.Bidirectional(layers.LSTM(LSTM_UNITS)),
        layers.Dropout(0.5),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(n_classes, activation="softmax"),
    ])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m


def bilstm_build_and_fit(train_texts, train_labels, classes, verbose=0):
    label2idx = {c: i for i, c in enumerate(classes)}
    y = train_labels.map(label2idx).values
    X_tr_text, X_val_text, y_tr, y_val = train_test_split(train_texts, y, test_size=0.15, random_state=SEED, stratify=y)

    tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
    tokenizer.fit_on_texts(X_tr_text)
    maxlen = int(np.percentile([len(t.split()) for t in X_tr_text], 95)) + 5
    vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)

    def pad(texts):
        return pad_sequences(tokenizer.texts_to_sequences(texts), maxlen=maxlen, padding="post", truncating="post")

    cw_arr = compute_class_weight("balanced", classes=np.unique(y_tr), y=y_tr)
    class_weight = {i: w for i, w in enumerate(cw_arr)}

    model = build_bilstm_model(vocab_size, maxlen, len(classes))
    es = callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
    history = model.fit(pad(X_tr_text), y_tr, validation_data=(pad(X_val_text), y_val),
                         epochs=60, batch_size=16, class_weight=class_weight, callbacks=[es], verbose=verbose)
    return {"model": model, "tokenizer": tokenizer, "maxlen": maxlen, "classes": list(classes), "history": history.history}


def bilstm_predict_proba(fitted, texts, classes):
    padded = pad_sequences(fitted["tokenizer"].texts_to_sequences(texts), maxlen=fitted["maxlen"], padding="post", truncating="post")
    proba = fitted["model"].predict(padded, verbose=0)
    if list(fitted["classes"]) != list(classes):
        proba = align_proba(proba, fitted["classes"], classes)
    return proba


def bilstm_save(fitted, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    import pickle
    fitted["model"].save(os.path.join(out_dir, "bilstm_model.keras"))
    with open(os.path.join(out_dir, "bilstm_tokenizer.pkl"), "wb") as f:
        pickle.dump(fitted["tokenizer"], f)
    with open(os.path.join(out_dir, "bilstm_meta.pkl"), "wb") as f:
        pickle.dump({"maxlen": fitted["maxlen"], "classes": fitted["classes"]}, f)
    return {"dir": out_dir}


def bilstm_size(fitted):
    tmp = "/tmp/_bilstm_size_check.keras"
    fitted["model"].save(tmp)
    size = os.path.getsize(tmp)
    os.remove(tmp)
    return size


BILSTM_CANDIDATE = {
    "name": "BiLSTM", "family": "bilstm", "runnable": True, "reason_not_runnable": None,
    "build_and_fit": lambda texts, labels, classes: bilstm_build_and_fit(texts, labels, classes, verbose=0),
    "predict_proba": bilstm_predict_proba, "save": bilstm_save, "model_size_bytes": bilstm_size,
}

# Démonstration illustrative
t0 = time.time()
fitted_bilstm_demo = bilstm_build_and_fit(dev_df["text"], dev_df["label"], CLASSES, verbose=2)
print(f"\nEntraînement BiLSTM : {time.time()-t0:.1f}s")
proba_bilstm_demo = bilstm_predict_proba(fitted_bilstm_demo, test_df["text"], CLASSES)
preds_bilstm_demo = [CLASSES[i] for i in proba_bilstm_demo.argmax(axis=1)]
print(f"Accuracy (démonstration, test final) : {accuracy_score(test_df['label'], preds_bilstm_demo):.3f}")
print(f"Macro-F1 (démonstration, test final) : {f1_score(test_df['label'], preds_bilstm_demo, average='macro'):.3f}")

## 8. CamemBERT

In [ ]:
class WeightedLossTrainer(Trainer if TRANSFORMERS_AVAILABLE else object):
    # Trainer HuggingFace standard, sauf que la cross-entropy est pondérée
    # par les poids de classe -- l'équivalent, pour cette famille de modèle,
    # du class_weight='balanced' utilisé côté TF-IDF et du class_weight
    # Keras côté BiLSTM.

    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


def make_transformer_build_and_fit(model_name, max_length=64, epochs=4):
    def build_and_fit(train_texts, train_labels, classes, output_dir="/tmp/hf_run"):
        if not TRANSFORMERS_AVAILABLE:
            raise RuntimeError("transformers/torch non installés dans cet environnement.")
        label2id = {c: i for i, c in enumerate(classes)}
        id2label = {i: c for c, i in label2id.items()}
        X_tr, X_val, y_tr, y_val = train_test_split(
            train_texts, train_labels.map(label2id).values, test_size=0.15, random_state=SEED, stratify=train_labels,
        )
        tokenizer = AutoTokenizer.from_pretrained(model_name)

        def make_ds(texts, labels):
            ds = Dataset.from_dict({"text": list(texts), "label": list(labels)})
            return ds.map(lambda b: tokenizer(b["text"], truncation=True, padding="max_length", max_length=max_length), batched=True)

        train_ds, val_ds = make_ds(X_tr, y_tr), make_ds(X_val, y_val)
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(classes), id2label=id2label, label2id=label2id)
        cw = torch.tensor(
                            compute_class_weight(
                                class_weight="balanced",
                                classes=np.asarray(range(len(classes))),
                                y=np.asarray(y_tr)
                            ),
                            dtype=torch.float32
                        )

        args = TrainingArguments(
            output_dir=output_dir, num_train_epochs=epochs, per_device_train_batch_size=16,
            per_device_eval_batch_size=32, learning_rate=2e-5, eval_strategy="epoch",
            save_strategy="no", report_to=[], seed=SEED, logging_steps=20,
        )
        trainer = WeightedLossTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, class_weights=cw)
        trainer.train()
        return {"trainer": trainer, "tokenizer": tokenizer, "classes": list(classes), "max_length": max_length}
    return build_and_fit


def transformer_predict_proba(fitted, texts, classes):
    tokenizer, model = fitted["tokenizer"], fitted["trainer"].model
    model.eval()
    enc = tokenizer(list(texts), truncation=True, padding=True, max_length=fitted["max_length"], return_tensors="pt")
    with torch.no_grad():
        proba = torch.softmax(model(**enc).logits, dim=-1).numpy()
    if list(fitted["classes"]) != list(classes):
        proba = align_proba(proba, fitted["classes"], classes)
    return proba


def transformer_save(fitted, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    fitted["trainer"].model.save_pretrained(out_dir)
    fitted["tokenizer"].save_pretrained(out_dir)
    return {"dir": out_dir}


def transformer_size(fitted):
    return sum(p.numel() * p.element_size() for p in fitted["trainer"].model.parameters())


def make_transformer_candidate(name, model_name):
    return {
        "name": name, "family": "transformer",
        "runnable": TRANSFORMERS_AVAILABLE,
        "reason_not_runnable": None if TRANSFORMERS_AVAILABLE else (
            "transformers/torch indisponibles dans cet environnement (pas d'accès à huggingface.co, "
            "installation de torch impossible faute d'espace disque). Code prêt à l'emploi."
        ),
        "build_and_fit": make_transformer_build_and_fit(model_name),
        "predict_proba": transformer_predict_proba, "save": transformer_save, "model_size_bytes": transformer_size,
        "hf_model_name": model_name,
    }


CAMEMBERT_CANDIDATE = make_transformer_candidate("CamemBERT", "camembert-base")
print(f"CamemBERT -> runnable={CAMEMBERT_CANDIDATE['runnable']}")
if not CAMEMBERT_CANDIDATE["runnable"]:
    print(f"  raison : {CAMEMBERT_CANDIDATE['reason_not_runnable']}")

## 9. DistilCamemBERT

In [ ]:
DISTILCAMEMBERT_CANDIDATE = make_transformer_candidate("DistilCamemBERT", "cmarkea/distilcamembert-base")
print(f"DistilCamemBERT -> runnable={DISTILCAMEMBERT_CANDIDATE['runnable']}")
if not DISTILCAMEMBERT_CANDIDATE["runnable"]:
    print(f"  raison : {DISTILCAMEMBERT_CANDIDATE['reason_not_runnable']}")

REGISTRY = TFIDF_CANDIDATES + [BILSTM_CANDIDATE, CAMEMBERT_CANDIDATE, DISTILCAMEMBERT_CANDIDATE]
print(f"\n{len(REGISTRY)} candidats enregistrés :")
for c in REGISTRY:
    print(f"  - {c['name']:32s} [{c['family']:11s}] {'exécutable' if c['runnable'] else 'NON exécutable ici'}")

## 10. Validation croisée

`StratifiedKFold` à 5 folds, **sur le dev set uniquement** (le test final reste intouché). Pour chaque candidat exécutable : Macro-F1 (moyenne + écart-type), accuracy, précision/rappel macro, F1 par classe, temps d'entraînement, temps d'inférence par message. Un candidat non exécutable (`runnable=False`) est recensé dans les résultats avec sa raison, sans jamais être appelé.

In [ ]:
def run_cv(registry, dev_df, classes, n_splits=5, seed=SEED, verbose=True):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    X_all, y_all = dev_df["text"].reset_index(drop=True), dev_df["label"].reset_index(drop=True)
    split_indices = list(skf.split(X_all, y_all))

    results = {}
    for candidate in registry:
        name = candidate["name"]
        if not candidate["runnable"]:
            results[name] = {"runnable": False, "family": candidate["family"], "reason": candidate["reason_not_runnable"]}
            if verbose:
                print(f"[{name}] non exécuté ici : {candidate['reason_not_runnable']}")
            continue

        fold_f1, fold_acc, fold_prec, fold_rec, fold_train_t, fold_infer_t, fold_pc = [], [], [], [], [], [], []
        for fold_i, (tr_idx, val_idx) in enumerate(split_indices, start=1):
            X_tr, X_val = X_all.iloc[tr_idx], X_all.iloc[val_idx]
            y_tr, y_val = y_all.iloc[tr_idx], y_all.iloc[val_idx]

            t0 = time.time()
            fitted = candidate["build_and_fit"](X_tr, y_tr, classes)
            train_time = time.time() - t0

            t0 = time.time()
            proba = candidate["predict_proba"](fitted, X_val, classes)
            infer_time = (time.time() - t0) / max(len(X_val), 1)

            preds = [classes[i] for i in np.argmax(proba, axis=1)]
            macro_f1 = f1_score(y_val, preds, average="macro", labels=classes, zero_division=0)
            acc = accuracy_score(y_val, preds)
            prec = precision_score(y_val, preds, average="macro", labels=classes, zero_division=0)
            rec = recall_score(y_val, preds, average="macro", labels=classes, zero_division=0)
            pc = dict(zip(classes, f1_score(y_val, preds, average=None, labels=classes, zero_division=0)))

            fold_f1.append(macro_f1); fold_acc.append(acc); fold_prec.append(prec); fold_rec.append(rec)
            fold_train_t.append(train_time); fold_infer_t.append(infer_time); fold_pc.append(pc)
            if verbose:
                print(f"[{name}] fold {fold_i}/{n_splits} -> macro-F1={macro_f1:.3f} acc={acc:.3f} (train {train_time:.1f}s)")

        results[name] = {
            "runnable": True, "family": candidate["family"],
            "macro_f1_mean": float(np.mean(fold_f1)), "macro_f1_std": float(np.std(fold_f1)),
            "accuracy_mean": float(np.mean(fold_acc)), "accuracy_std": float(np.std(fold_acc)),
            "precision_mean": float(np.mean(fold_prec)), "recall_mean": float(np.mean(fold_rec)),
            "train_time_mean_s": float(np.mean(fold_train_t)),
            "infer_time_per_msg_ms": float(np.mean(fold_infer_t)) * 1000,
            "per_class_f1_mean": {c: float(np.mean([d[c] for d in fold_pc])) for c in classes},
            "fold_macro_f1": [float(x) for x in fold_f1],
        }
    return results


cv_results = run_cv(REGISTRY, dev_df, CLASSES, n_splits=5)
with open(f"{ARTIFACTS_DIR}/cv_results.json", "w", encoding="utf-8") as f:
    json.dump(cv_results, f, ensure_ascii=False, indent=2)

## 11. Comparaison des modèles

In [ ]:
def results_to_markdown(results, classes):
    runnable = {k: v for k, v in results.items() if v["runnable"]}
    not_runnable = {k: v for k, v in results.items
     () if not v["runnable"]}

    lines = ["| Modèle | Macro-F1 (moy ± ET) | Accuracy | Précision (macro) | Rappel (macro) | Temps entraînement | Temps inférence/msg |",
             "|---|---|---|---|---|---|---|"]
    for name, r in sorted(runnable.items(), key=lambda kv: -kv[1]["macro_f1_mean"]):
        lines.append(f"| {name} | {r['macro_f1_mean']:.3f} ± {r['macro_f1_std']:.3f} | {r['accuracy_mean']:.3f} | "
                      f"{r['precision_mean']:.3f} | {r['recall_mean']:.3f} | {r['train_time_mean_s']:.1f} s | {r['infer_time_per_msg_ms']:.2f} ms |")
    for name, r in not_runnable.items():
        lines.append(f"| {name} | *non exécuté* | – | – | – | – | – |")
    md_table = "\n".join(lines)

    md_table += "\n\n**F1 par classe (candidats exécutés) :**\n\n"
    rows = ["| Modèle | " + " | ".join(classes) + " |", "|---|" + "---|" * len(classes)]
    for name, r in sorted(runnable.items(), key=lambda kv: -kv[1]["macro_f1_mean"]):
        rows.append(f"| {name} | " + " | ".join(f"{r['per_class_f1_mean'][c]:.2f}" for c in classes) + " |")
    return md_table + "\n".join(rows)


from IPython.display import Markdown, display
display(Markdown(results_to_markdown(cv_results, CLASSES)))

In [ ]:
runnable_results = {k: v for k, v in cv_results.items() if v["runnable"]}
fig, ax = plt.subplots(figsize=(8, 4.5))
names = list(runnable_results.keys())
means = [runnable_results[n]["macro_f1_mean"] for n in names]
stds = [runnable_results[n]["macro_f1_std"] for n in names]
order = np.argsort(means)[::-1]
names, means, stds = [names[i] for i in order], [means[i] for i in order], [stds[i] for i in order]
colors = ["#E8A33D" if i == 0 else "#4C72B0" for i in range(len(names))]
ax.bar(names, means, yerr=stds, capsize=5, color=colors)
ax.set_ylabel("Macro-F1 (validation croisée, 5 folds)")
ax.set_title("Comparaison des modèles exécutés")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 12. Sélection automatique du meilleur modèle

Sélection au **Macro-F1 moyen** (jamais à l'accuracy, qui favoriserait mécaniquement un modèle qui se contente de bien prédire les classes majoritaires). Cette fonction ne contient aucun nom de modèle en dur : elle prend le `max()` sur les résultats réellement obtenus, donc un autre corpus qui ferait gagner un autre modèle serait automatiquement.

In [ ]:
def select_best(cv_results):
    runnable = {name: r for name, r in cv_results.items() if r["runnable"]}
    if not runnable:
        raise RuntimeError("Aucun candidat exécutable : impossible de sélectionner un modèle.")
    return max(runnable, key=lambda n: runnable[n]["macro_f1_mean"])


best_name = select_best(cv_results)
print(f">>> Modèle sélectionné automatiquement (Macro-F1 moyen le plus élevé) : {best_name}")
print(f"    Macro-F1 CV : {cv_results[best_name]['macro_f1_mean']:.3f} ± {cv_results[best_name]['macro_f1_std']:.3f}")

REGISTRY_BY_NAME = {c["name"]: c for c in REGISTRY}

## 13. Réentraînement final

Le **type** de modèle gagnant est réentraîné sur l'intégralité du jeu de développement (les 80% utilisés pour la validation croisée, mais cette fois sans découpage en folds) .

In [ ]:
def retrain_final(registry_by_name, best_name, dev_df, classes):
    candidate = registry_by_name[best_name]
    fitted = candidate["build_and_fit"](dev_df["text"], dev_df["label"], classes)
    return candidate, fitted


print(f"Réentraînement de '{best_name}' sur l'intégralité du dev set ({len(dev_df)} messages)...")
t0 = time.time()
final_candidate, final_fitted = retrain_final(REGISTRY_BY_NAME, best_name, dev_df, CLASSES)
print(f"Terminé en {time.time()-t0:.1f}s")

## 14. Évaluation sur le test final

In [ ]:
proba_test = final_candidate["predict_proba"](final_fitted, test_df["text"], CLASSES)
preds_test = [CLASSES[i] for i in np.argmax(proba_test, axis=1)]
y_true = test_df["label"].tolist()

test_metrics = {
    "accuracy": float(accuracy_score(y_true, preds_test)),
    "macro_f1": float(f1_score(y_true, preds_test, average="macro", labels=CLASSES, zero_division=0)),
    "weighted_f1": float(f1_score(y_true, preds_test, average="weighted", labels=CLASSES, zero_division=0)),
    "n_test": len(test_df),
}
report = classification_report(y_true, preds_test, labels=CLASSES, digits=3, zero_division=0)

print(f"=== Résultats du modèle sélectionné ({best_name}) sur le test final ===")
print(f"Accuracy    : {test_metrics['accuracy']:.3f}")
print(f"Macro-F1    : {test_metrics['macro_f1']:.3f}")
print(f"Weighted-F1 : {test_metrics['weighted_f1']:.3f}\n")
print(report)

display(Markdown(
    "| Métrique | Valeur |\n|---|---|\n"
    f"| Modèle sélectionné | **{best_name}** |\n"
    f"| Accuracy (test final) | {test_metrics['accuracy']:.3f} |\n"
    f"| Macro-F1 (test final) | {test_metrics['macro_f1']:.3f} |\n"
    f"| Weighted-F1 (test final) | {test_metrics['weighted_f1']:.3f} |\n"
    f"| Taille du test final | {test_metrics['n_test']} messages |"
))

In [ ]:
cm = confusion_matrix(y_true, preds_test, labels=CLASSES)
fig, ax = plt.subplots(figsize=(7.5, 6.5))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
ax.set_title(f"Matrice de confusion — {best_name} (test final)")
plt.tight_layout()
plt.show()

## 15. Sauvegarde du modèle

In [ ]:
final_model_dir = f"{ARTIFACTS_DIR}/final_model"
saved_paths = final_candidate["save"](final_fitted, final_model_dir)
print("Modèle final sauvegardé :", saved_paths)

manifest_selection = {
    "selected_model": best_name,
    "family": final_candidate["family"],
    "selection_criterion": "macro_f1_mean (validation croisée 5 folds sur le dev set)",
    "cv_macro_f1_mean": cv_results[best_name]["macro_f1_mean"],
    "cv_macro_f1_std": cv_results[best_name]["macro_f1_std"],
    "test_metrics": test_metrics,
    "classes": CLASSES,
}
with open(f"{ARTIFACTS_DIR}/selection_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest_selection, f, ensure_ascii=False, indent=2)
print(f"\nManifeste de sélection sauvegardé -> {ARTIFACTS_DIR}/selection_manifest.json")